# Test 05: All Languages (R + Scala + Julia)

The full integration test. Installs all three languages in a single
shared conda environment and verifies they all work together.

**What this tests:**
1. All three language plugins install without package conflicts
2. Shared environment has R, Java, and Julia binaries
3. All three magics (`%%R`, `%%scala`, `%%julia`) work in one notebook
4. Four-language pipeline: Python -> R -> Python -> Scala -> Python -> Julia
5. All three can access Snowflake

**Prerequisites:**
- Full EAI with conda-forge, CRAN, Maven, GitHub, PyPI access
- Expect ~8-15 min on first install (all three languages)

## 1. Install All Languages

In [ ]:
# Install from local source (notebooks/ is inside the package dir)
!pip install -q ..

In [ ]:
from sfnb_multilang import install

report = install(languages=["r", "scala", "julia"])

In [ ]:
print(f"Overall success: {report.success}")
print(f"Duration: {report.elapsed_seconds:.1f}s")
print(f"Env prefix: {report.env_prefix}")
print()

for lang in ["r", "scala", "julia"]:
    pr = report.plugin_results.get(lang)
    vr = report.validation_results.get(lang)
    if pr:
        ok = pr.success and (vr.success if vr else False)
        print(f"  {lang:8s} {'PASS' if ok else 'FAIL':6s} {pr.version}")
        if pr.errors:
            for e in pr.errors:
                print(f"           ERROR: {e}")

## 2. Verify Shared Environment

All three runtimes should coexist in one conda prefix.

In [ ]:
import os

prefix = report.env_prefix
binaries = {
    "R": os.path.join(prefix, "bin", "R"),
    "java": os.path.join(prefix, "bin", "java"),
    "julia": os.path.join(prefix, "bin", "julia"),
}

all_ok = True
for name, path in binaries.items():
    exists = os.path.isfile(path)
    status = "OK" if exists else "MISSING"
    print(f"  {name:8s} {status:8s} {path}")
    if not exists:
        all_ok = False

assert all_ok, "FAIL: One or more binaries missing from shared env"
print("\nPASS: All three runtimes in shared environment")

## 3. Set Up R and Scala Magics

Register `%%R` and `%%scala` magics first. Julia is initialised later
to avoid JVM/Julia signal handler conflicts (SIGSEGV).

In [ ]:
from r_helpers import setup_r_environment
setup_r_environment()

In [ ]:
from scala_helpers import setup_scala_environment
setup_scala_environment()

In [ ]:
from snowflake.snowpark.context import get_active_session
from scala_helpers import bootstrap_snowpark_scala

session = get_active_session()
ok, msg = bootstrap_snowpark_scala(session)
print(f"Scala Snowpark bootstrap: {'OK' if ok else 'FAIL'}")
if msg:
    print(msg)

## 4. Set Up Julia Magic

Julia is initialised after Scala Snowpark bootstrap to prevent
signal handler conflicts between the JVM and Julia runtime.

In [ ]:
from julia_helpers import setup_julia_environment
setup_julia_environment()

## 5. Smoke Tests -- All Three Languages

In [ ]:
%%R
cat("R: ", R.version.string, "\n")
library(dplyr)
cat("PASS: R smoke test\n")

In [ ]:
%%scala
println(s"Scala: ${scala.util.Properties.versionString}")
println(s"Java: ${System.getProperty("java.version")}")
println("PASS: Scala smoke test")

In [ ]:
%%julia
println("Julia: ", VERSION)
using DataFrames
println("PASS: Julia smoke test")

## 6. Four-Language Data Pipeline

Demonstrating data flowing across all four environments:
Python -> R (aggregate) -> Python -> Scala (enrich) -> Python -> Julia (analyse)

In [ ]:
import pandas as pd

# Step 1: Create data in Python
sales = pd.DataFrame({
    "region": ["North", "South", "North", "South", "North", "South"],
    "quarter": ["Q1", "Q1", "Q2", "Q2", "Q3", "Q3"],
    "revenue": [100, 80, 120, 90, 140, 110]
})
print("Step 1 - Raw data (Python):")
print(sales)

In [ ]:
%%R -i sales -o r_agg
# Step 2: Aggregate in R
library(dplyr)
r_agg <- sales %>%
  group_by(region) %>%
  summarise(total = sum(revenue), avg = mean(revenue))
r_agg

In [ ]:
# Step 3: Python receives R result and extracts scalars for other languages
print("Step 3 - R result in Python:")
print(r_agg)
assert len(r_agg) == 2, "Expected 2 regions"
total_revenue = float(r_agg["total"].sum())
num_regions = len(r_agg)
print(f"Extracted: {num_regions} regions, total revenue = {total_revenue}")

In [ ]:
%%scala -i total_revenue,num_regions
// Step 4: Scala receives scalar values (R -> Python -> Scala)
val rev = total_revenue.asInstanceOf[Double]
val regions = num_regions.asInstanceOf[Int]
println(s"Step 4 - Scala: $regions regions, total revenue = $rev")
println("PASS: Scala received R-aggregated data via Python scalars")

In [ ]:
%%julia -i total_revenue,num_regions
# Step 5: Julia receives scalar values (R -> Python -> Julia)
println("Step 5 - Julia: $(Int(num_regions)) regions, total revenue = $(total_revenue)")
println("PASS: Julia received R-aggregated data via Python scalars")

In [ ]:
print("="*60)
print("PASS: Four-language pipeline complete!")
print("  Python -> R -> Python -> Scala -> Python -> Julia")
print("="*60)

## 7. All Languages Query Snowflake

Session and Scala Snowpark were bootstrapped in Section 3.

In [ ]:
sf_info = session.sql("SELECT CURRENT_ACCOUNT() AS acct").to_pandas()
print(f"Python: account = {sf_info['ACCT'][0]}")

In [ ]:
%%R -i sf_info
cat("R: account =", sf_info$ACCT, "\n")

In [ ]:
%%scala
val acct = sfSession.sql("SELECT CURRENT_ACCOUNT()").collect()(0).getString(0)
println(s"Scala: account = $acct")
println("PASS: Scala Snowflake connectivity")

In [ ]:
%%julia -i sf_info
using DataFrames
df = DataFrame(sf_info)
println("Julia: account = ", df.ACCT[1])
println("PASS: Julia Snowflake connectivity")

In [ ]:
print("="*60)
print("PASS: All three languages query Snowflake successfully")
print("="*60)

## Test Summary

| Test | Expected |
|---|---|
| All-language install | 3 plugins in report, all success |
| Shared env | R, java, julia binaries in same prefix |
| R magic | `%%R` works |
| Scala magic | `%%scala` works |
| Julia magic | `%%julia` works |
| Pipeline | Python -> R -> Python -> Scala -> Python -> Julia |
| Snowflake | All three query the active session |